# Upload the frozen R2 adapter to the private GitHub Release

Run only during packaging. This notebook never reads leaderboard or final-test questions and is not part of official inference. It uploads the verified 119.8MB LoRA weight as a release asset because ordinary Git rejects files over 100MB.

In [ ]:
# Cell 1 — Mount Drive and verify the exact frozen adapter.
import hashlib
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
ADAPTER=Path('/content/drive/MyDrive/2026소중한챌린지/runs/RFT-0004B-r2-pro4-hint-lowdrift-lora/adapter_final')
WEIGHT=ADAPTER/'adapter_model.safetensors'
EXPECTED='e4a22286b3b6a3108c0f2a374012601309abee6511b96b2a108749d432909f11'
assert (ADAPTER/'adapter_config.json').exists() and WEIGHT.exists(),ADAPTER
h=hashlib.sha256()
with WEIGHT.open('rb') as f:
    for b in iter(lambda:f.read(4*1024*1024),b''):h.update(b)
assert h.hexdigest()==EXPECTED,h.hexdigest()
print('[ADAPTER VERIFIED]',WEIGHT,WEIGHT.stat().st_size)

In [ ]:
# Cell 2 — Install GitHub CLI, then approve its one-time device-login code in the browser.
!type -p gh >/dev/null || (curl -fsSL https://cli.github.com/packages/githubcli-archive-keyring.gpg | dd of=/usr/share/keyrings/githubcli-archive-keyring.gpg && chmod go+r /usr/share/keyrings/githubcli-archive-keyring.gpg && echo 'deb [arch=$(dpkg --print-architecture) signed-by=/usr/share/keyrings/githubcli-archive-keyring.gpg] https://cli.github.com/packages stable main' | tee /etc/apt/sources.list.d/github-cli.list >/dev/null && apt-get update -qq && apt-get install -y gh)
!gh auth login -h github.com --web --git-protocol https

In [ ]:
# Cell 3 — Clone the private repository and upload the release asset.
import subprocess,sys
REPO=Path('/content/qwen-math-final-2026')
if not REPO.exists(): subprocess.run(['git','clone','https://github.com/jhparktime/qwen-math-final-2026.git',str(REPO)],check=True)
else: subprocess.run(['git','-C',str(REPO),'pull','--ff-only'],check=True)
subprocess.run([sys.executable,str(REPO/'scripts/upload_adapter_release.py'),'--adapter-dir',str(ADAPTER)],cwd=REPO,check=True)

In [ ]:
# Final cell — Optional runtime release.
DISCONNECT_GPU_RUNTIME=False
if DISCONNECT_GPU_RUNTIME:
    from google.colab import runtime
    runtime.unassign()
else:
    print('[RUNTIME] retained')